In [1]:
!pip install torch torchvision transformers scikit-learn pandas tqdm

In [2]:
import os
os.listdir("/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data")

['dev.jsonl', 'test.jsonl', 'README.md', 'LICENSE.txt', 'train.jsonl', 'img']

In [3]:
import pandas as pd

data_path = "/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data"

df = pd.read_json(
    f"{data_path}/train.jsonl",
    lines=True
)

print("Dataset size:", len(df))
df.head()

Dataset size: 8500


,id,img,label,text
0,42953,img/42953.png,0,its their character not their color that matters
1,23058,img/23058.png,0,don't be afraid to love again everyone is not ...
2,13894,img/13894.png,0,putting bows on your pet
3,37408,img/37408.png,0,i love everything and everybody! except for sq...
4,82403,img/82403.png,0,"everybody loves chocolate chip cookies, even h..."


In [4]:
import torch
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [5]:
data_path = "/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data"

# Create full image path
df["image_path"] = df["img"].apply(
    lambda x: f"{data_path}/{x}"
)

# Keep only necessary columns
df = df[["text", "image_path", "label"]]

df.head()

,text,image_path,label
0,its their character not their color that matters,/kaggle/input/datasets/parthplc/facebook-hatef...,0
1,don't be afraid to love again everyone is not ...,/kaggle/input/datasets/parthplc/facebook-hatef...,0
2,putting bows on your pet,/kaggle/input/datasets/parthplc/facebook-hatef...,0
3,i love everything and everybody! except for sq...,/kaggle/input/datasets/parthplc/facebook-hatef...,0
4,"everybody loves chocolate chip cookies, even h...",/kaggle/input/datasets/parthplc/facebook-hatef...,0


In [6]:
from tqdm import tqdm
import torch

def get_text_embeddings(texts, batch_size=32):

    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):

        batch = texts[i:i+batch_size]

        inputs = processor(
            text=batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():

            outputs = model.get_text_features(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"]
            )

            # ensure tensor
            if hasattr(outputs, "pooler_output"):
                outputs = outputs.pooler_output

        embeddings.append(outputs.cpu())

    embeddings = torch.cat(embeddings, dim=0)

    return embeddings.numpy()

In [7]:
from PIL import Image

def get_image_embeddings(paths, batch_size=32):

    embeddings = []

    for i in tqdm(range(0, len(paths), batch_size)):

        batch_paths = paths[i:i+batch_size]

        images = [Image.open(p).convert("RGB") for p in batch_paths]

        inputs = processor(
            images=images,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():

            outputs = model.get_image_features(
                pixel_values=inputs["pixel_values"]
            )

            if hasattr(outputs, "pooler_output"):
                outputs = outputs.pooler_output

        embeddings.append(outputs.cpu())

    embeddings = torch.cat(embeddings, dim=0)

    return embeddings.numpy()

In [8]:
texts = df["text"].tolist()
images = df["image_path"].tolist()

text_emb = get_text_embeddings(texts)
img_emb = get_image_embeddings(images)

print(text_emb.shape)
print(img_emb.shape)

100%|██████████| 266/266 [04:03<00:00,  1.09it/s]

(8500, 512)
(8500, 512)


In [9]:
import numpy as np

text_emb = text_emb / np.linalg.norm(text_emb, axis=1, keepdims=True)
img_emb = img_emb / np.linalg.norm(img_emb, axis=1, keepdims=True)

In [10]:
X = np.concatenate([
    text_emb,
    img_emb
], axis=1)

y = df["label"].values

In [11]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

pos_weight = neg / pos

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    tree_method="hist"
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [12]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.75      0.79      0.77      1090
           1       0.59      0.53      0.56       610

    accuracy                           0.70      1700
   macro avg       0.67      0.66      0.66      1700
weighted avg       0.69      0.70      0.69      1700

ROC-AUC: 0.7410648217777109


In [13]:
import numpy as np

# Save predictions
np.save("/kaggle/working/hateful_y_pred.npy", y_pred)
np.save("/kaggle/working/hateful_y_test.npy", y_test)

# Save prediction probabilities
np.save("/kaggle/working/hateful_y_prob.npy", y_prob)

# Save embeddings (optional but useful)
np.save("/kaggle/working/hateful_text_emb.npy", text_emb)
np.save("/kaggle/working/hateful_img_emb.npy", img_emb)

print("Saved all results for Notebook 9")

Saved all results for Notebook 9
